In [1]:
from pathlib import Path
import pandas as pd

print("02.22 STEP 1 - INSPECT TEXT/CATEGORY DATA")
print("=" * 60)

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
BASE_DIR = Path.cwd().parent

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "outlier_adjusted"
    / "sales_adjusted"
)

PRODUCT_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "products_master.csv"
)

# ---------------------------------------------------------
# LOAD SALES FILES
# ---------------------------------------------------------
sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

print()
print("Sales files found:", len(sales_files))

# Read all sales files
sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print("Total sales rows:", len(sales))

# ---------------------------------------------------------
# LOAD PRODUCT MASTER
# ---------------------------------------------------------
products = pd.read_csv(PRODUCT_FILE)

print("Product master rows:", len(products))

# ---------------------------------------------------------
# SALES TEXT/CATEGORY FIELDS
# ---------------------------------------------------------
print()
print("Sales text fields")
print("-" * 60)

sales_text_columns = [
    "channel",
    "payment_method",
    "status"
]

for col in sales_text_columns:
    if col in sales.columns:
        print()
        print(f"{col}:")
        print("Unique values:", sales[col].nunique(dropna=False))
        print(sales[col].value_counts(dropna=False).head(20))

# ---------------------------------------------------------
# PRODUCT CATEGORY FIELDS
# ---------------------------------------------------------
print()
print("Product category fields")
print("-" * 60)

product_category_columns = [
    "category",
    "subcategory",
    "brand",
    "status"
]

for col in product_category_columns:
    if col in products.columns:
        print()
        print(f"{col}:")
        print("Unique values:", products[col].nunique(dropna=False))
        print(products[col].value_counts(dropna=False).head(20))

# ---------------------------------------------------------
# PRODUCT CATEGORY LINKAGE
# ---------------------------------------------------------
print()
print("Sales-to-product category linkage")
print("-" * 60)

if "product_id" in sales.columns and "product_id" in products.columns:

    sales_product = sales[
        ["product_id"]
    ].merge(
        products[
            ["product_id", "category", "subcategory", "brand"]
        ],
        on="product_id",
        how="left"
    )

    print(
        "Sales rows with category:",
        sales_product["category"].notna().sum()
    )

    print(
        "Sales rows without category:",
        sales_product["category"].isna().sum()
    )

    print()
    print("Category distribution in sales:")
    print(
        sales_product["category"]
        .value_counts(dropna=False)
    )

# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------
print()
print("02.22 Step 1 inspection: COMPLETED")

02.22 STEP 1 - INSPECT TEXT/CATEGORY DATA

Sales files found: 32
Total sales rows: 2118200
Product master rows: 2500

Sales text fields
------------------------------------------------------------

payment_method:
Unique values: 7
payment_method
Debit Card         371440
Wallet             370467
UPI                370398
Net Banking        370179
Credit Card        369548
Cash               265492
UNKNOWN_PAYMENT       676
Name: count, dtype: int64

Product category fields
------------------------------------------------------------

category:
Unique values: 5
category
Fashion                   509
Beauty & Personal Care    505
Home & Kitchen            504
Electronics               500
Grocery                   482
Name: count, dtype: int64

subcategory:
Unique values: 25
subcategory
Mobile          116
Cookware        116
Jacket          114
Perfume         112
Laptop          112
Dining Table    110
Kurti           107
Shampoo         106
Shirt           106
Wheat Flour     104
Coo

In [2]:
print("02.22 STEP 2 - DEFINE INCONSISTENT TEXT/CATEGORY SCENARIO")
print("=" * 60)

# ---------------------------------------------------------
# BASELINE
# ---------------------------------------------------------
TOTAL_SALES_ROWS = 2_118_200

# ---------------------------------------------------------
# APPROVED SIMULATION RATES
# From Document 03:
# Whitespace/case inconsistency ≈ 0.40%
# Spelling variation / typo ≈ 0.15%
# ---------------------------------------------------------
WHITESPACE_CASE_RATE = 0.004
SPELLING_VARIATION_RATE = 0.0015

# Calculate controlled scenario targets
TARGET_WHITESPACE_CASE = round(
    TOTAL_SALES_ROWS * WHITESPACE_CASE_RATE
)

TARGET_SPELLING_VARIATION = round(
    TOTAL_SALES_ROWS * SPELLING_VARIATION_RATE
)

# ---------------------------------------------------------
# SCENARIO DEFINITIONS
# ---------------------------------------------------------
SCENARIOS = {
    "whitespace_case": {
        "rate": WHITESPACE_CASE_RATE,
        "target_rows": TARGET_WHITESPACE_CASE,
        "description": "Controlled whitespace and case inconsistencies",
        "examples": [
            " Hyderabad ",
            "HYDERABAD",
            "fashion",
            "upi "
        ]
    },

    "spelling_variation": {
        "rate": SPELLING_VARIATION_RATE,
        "target_rows": TARGET_SPELLING_VARIATION,
        "description": "Controlled spelling variations and typos",
        "examples": [
            "Electrnics",
            "Bengaluruu",
            "Quailty Issue"
        ]
    }
}

# ---------------------------------------------------------
# DISPLAY SCENARIO
# ---------------------------------------------------------
print()
print("Inconsistent Text/Category Injection Targets")
print("-" * 60)

for scenario, details in SCENARIOS.items():

    print(f"\n{scenario}")
    print(f"Rate: {details['rate'] * 100:.2f}%")
    print(f"Target rows: {details['target_rows']:,}")
    print(f"Description: {details['description']}")
    print("Examples:")

    for example in details["examples"]:
        print(f"  - {example}")

# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------
rates_valid = (
    WHITESPACE_CASE_RATE == 0.004
    and SPELLING_VARIATION_RATE == 0.0015
)

targets_valid = (
    TARGET_WHITESPACE_CASE > 0
    and TARGET_SPELLING_VARIATION > 0
)

if rates_valid and targets_valid:
    print()
    print("02.22 Step 2 validation: PASSED")
else:
    print()
    print("02.22 Step 2 validation: FAILED")

02.22 STEP 2 - DEFINE INCONSISTENT TEXT/CATEGORY SCENARIO

Inconsistent Text/Category Injection Targets
------------------------------------------------------------

whitespace_case
Rate: 0.40%
Target rows: 8,473
Description: Controlled whitespace and case inconsistencies
Examples:
  -  Hyderabad 
  - HYDERABAD
  - fashion
  - upi 

spelling_variation
Rate: 0.15%
Target rows: 3,177
Description: Controlled spelling variations and typos
Examples:
  - Electrnics
  - Bengaluruu
  - Quailty Issue

02.22 Step 2 validation: PASSED


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.22 STEP 3 - INJECT CONTROLLED INCONSISTENCIES")
print("=" * 60)

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
BASE_DIR = Path.cwd().parent

SALES_INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "outlier_adjusted"
    / "sales_adjusted"
)

OUTPUT_BASE_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "inconsistent_text_category_adjusted"
)

SALES_OUTPUT_DIR = OUTPUT_BASE_DIR / "sales_adjusted"

SALES_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# MASTER FILES
# ---------------------------------------------------------
PRODUCT_MASTER_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "products_master.csv"
)

STORE_MASTER_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "stores_master.csv"
)

PRODUCT_OUTPUT_FILE = (
    OUTPUT_BASE_DIR
    / "products_master_adjusted.csv"
)

STORE_OUTPUT_FILE = (
    OUTPUT_BASE_DIR
    / "stores_master_adjusted.csv"
)

# ---------------------------------------------------------
# APPROVED TARGETS
# ---------------------------------------------------------
TARGET_WHITESPACE_CASE = 8473
TARGET_SPELLING_VARIATION = 3177

# ---------------------------------------------------------
# RANDOM GENERATOR
# ---------------------------------------------------------
rng = np.random.default_rng(2202)

# =========================================================
# PART 1 - SALES PAYMENT METHOD INCONSISTENCIES
# =========================================================

print()
print("PART 1 - SALES PAYMENT METHOD INCONSISTENCIES")
print("-" * 60)

input_files = sorted(
    SALES_INPUT_DIR.glob("sales_*.csv")
)

if len(input_files) != 32:
    raise ValueError(
        f"Expected 32 sales files, found {len(input_files)}"
    )

sales_parts = []

for file in input_files:

    df = pd.read_csv(file)

    if "payment_method" not in df.columns:
        raise ValueError(
            f"'payment_method' missing from {file.name}"
        )

    sales_parts.append(df)

sales = pd.concat(
    sales_parts,
    ignore_index=True
)

original_sales_rows = len(sales)

print(f"Sales files found: {len(input_files)}")
print(f"Total sales rows: {original_sales_rows:,}")

# ---------------------------------------------------------
# ELIGIBLE PAYMENT METHODS
# ---------------------------------------------------------
eligible_payment = sales.index[
    sales["payment_method"].isin(
        [
            "UPI",
            "Debit Card",
            "Credit Card",
            "Net Banking",
            "Wallet"
        ]
    )
].to_numpy()

if len(eligible_payment) < TARGET_WHITESPACE_CASE:
    raise ValueError(
        "Not enough eligible payment-method rows."
    )

selected_payment = rng.choice(
    eligible_payment,
    size=TARGET_WHITESPACE_CASE,
    replace=False
)

# ---------------------------------------------------------
# CONTROLLED CASE / WHITESPACE VARIATIONS
# ---------------------------------------------------------
variation_map = {
    "UPI": "upi ",
    "Debit Card": " DEBIT CARD",
    "Credit Card": "credit card",
    "Net Banking": "Net  Banking",
    "Wallet": " wallet "
}

for idx in selected_payment:

    original_value = sales.at[
        idx,
        "payment_method"
    ]

    sales.at[
        idx,
        "payment_method"
    ] = variation_map[original_value]

print(
    f"Payment-method inconsistencies injected: "
    f"{len(selected_payment):,}"
)

# =========================================================
# SAVE SALES FILES
# =========================================================

start = 0

for file in input_files:

    original_df = pd.read_csv(file)

    row_count = len(original_df)

    adjusted_df = sales.iloc[
        start:start + row_count
    ].copy()

    output_file = SALES_OUTPUT_DIR / file.name

    adjusted_df.to_csv(
        output_file,
        index=False
    )

    start += row_count

# =========================================================
# PART 2 - PRODUCT CATEGORY SPELLING INCONSISTENCY
# =========================================================

print()
print("PART 2 - PRODUCT CATEGORY INCONSISTENCY")
print("-" * 60)

products = pd.read_csv(
    PRODUCT_MASTER_FILE
)

original_product_rows = len(products)

if "category" not in products.columns:
    raise ValueError(
        "'category' column missing from products_master.csv"
    )

eligible_products = products.index[
    products["category"].eq("Electronics")
].to_numpy()

if len(eligible_products) == 0:
    raise ValueError(
        "No Electronics products found."
    )

product_target = min(
    TARGET_SPELLING_VARIATION,
    len(eligible_products)
)

selected_products = rng.choice(
    eligible_products,
    size=product_target,
    replace=False
)

products.loc[
    selected_products,
    "category"
] = "Electrnics"

products.to_csv(
    PRODUCT_OUTPUT_FILE,
    index=False
)

print(
    f"Product category spelling inconsistencies: "
    f"{product_target:,}"
)

print(
    f"Products rows preserved: "
    f"{len(products):,}"
)

# =========================================================
# PART 3 - STORE CITY SPELLING INCONSISTENCY
# =========================================================

print()
print("PART 3 - STORE CITY INCONSISTENCY")
print("-" * 60)

stores = pd.read_csv(
    STORE_MASTER_FILE
)

original_store_rows = len(stores)

if "city" not in stores.columns:
    raise ValueError(
        "'city' column missing from stores_master.csv"
    )

eligible_stores = stores.index[
    stores["city"].eq("Bengaluru")
].to_numpy()

if len(eligible_stores) > 0:

    stores.loc[
        eligible_stores,
        "city"
    ] = "Bengaluruu"

store_city_changes = len(
    eligible_stores
)

stores.to_csv(
    STORE_OUTPUT_FILE,
    index=False
)

print(
    f"Store city spelling inconsistencies: "
    f"{store_city_changes}"
)

print(
    f"Stores rows preserved: "
    f"{len(stores):,}"
)

# =========================================================
# FINAL VALIDATION
# =========================================================

output_files = sorted(
    SALES_OUTPUT_DIR.glob("sales_*.csv")
)

output_sales_rows = sum(
    len(pd.read_csv(file))
    for file in output_files
)

print()
print("FINAL VALIDATION")
print("-" * 60)

print(
    f"Sales files created: {len(output_files)}"
)

print(
    f"Original sales rows: {original_sales_rows:,}"
)

print(
    f"Output sales rows: {output_sales_rows:,}"
)

print(
    f"Product master rows: {len(products):,}"
)

print(
    f"Store master rows: {len(stores):,}"
)

file_count_valid = (
    len(output_files) == 32
)

sales_rows_valid = (
    original_sales_rows == output_sales_rows
)

product_rows_valid = (
    original_product_rows == len(products)
)

store_rows_valid = (
    original_store_rows == len(stores)
)

if (
    file_count_valid
    and sales_rows_valid
    and product_rows_valid
    and store_rows_valid
):
    print()
    print("02.22 Step 3 injection: PASSED")
else:
    print()
    print("02.22 Step 3 injection: FAILED")

02.22 STEP 3 - INJECT CONTROLLED INCONSISTENCIES

PART 1 - SALES PAYMENT METHOD INCONSISTENCIES
------------------------------------------------------------
Sales files found: 32
Total sales rows: 2,118,200
Payment-method inconsistencies injected: 8,473

PART 2 - PRODUCT CATEGORY INCONSISTENCY
------------------------------------------------------------
Product category spelling inconsistencies: 500
Products rows preserved: 2,500

PART 3 - STORE CITY INCONSISTENCY
------------------------------------------------------------
Store city spelling inconsistencies: 4
Stores rows preserved: 60

FINAL VALIDATION
------------------------------------------------------------
Sales files created: 32
Original sales rows: 2,118,200
Output sales rows: 2,118,200
Product master rows: 2,500
Store master rows: 60

02.22 Step 3 injection: PASSED


In [5]:
from pathlib import Path
import pandas as pd

print("02.22 STEP 4 - VALIDATE INCONSISTENT TEXT/CATEGORY SCENARIOS")
print("=" * 65)

BASE_DIR = Path.cwd().parent

OUTPUT_BASE_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "inconsistent_text_category_adjusted"
)

SALES_OUTPUT_DIR = OUTPUT_BASE_DIR / "sales_adjusted"

PRODUCT_OUTPUT_FILE = (
    OUTPUT_BASE_DIR
    / "products_master_adjusted.csv"
)

STORE_OUTPUT_FILE = (
    OUTPUT_BASE_DIR
    / "stores_master_adjusted.csv"
)

# ---------------------------------------------------------
# EXPECTED VALUES
# ---------------------------------------------------------

EXPECTED_FILES = 32
EXPECTED_SALES_ROWS = 2_118_200
EXPECTED_PAYMENT_INCONSISTENCIES = 8_473
EXPECTED_PRODUCT_SPELLING = 500
EXPECTED_STORE_SPELLING = 4

# ---------------------------------------------------------
# VALIDATE SALES FILES
# ---------------------------------------------------------

sales_files = sorted(
    SALES_OUTPUT_DIR.glob("sales_*.csv")
)

total_sales_rows = 0
payment_inconsistency_count = 0

for file in sales_files:

    df = pd.read_csv(file)

    total_sales_rows += len(df)

    payment_inconsistency_count += (
        df["payment_method"]
        .astype(str)
        .str.contains(
            r"^\s|\s$|  |^[a-z]|^[A-Z ]+$",
            regex=True
        )
        .sum()
    )

# ---------------------------------------------------------
# VALIDATE PRODUCT MASTER
# ---------------------------------------------------------

products = pd.read_csv(
    PRODUCT_OUTPUT_FILE
)

product_spelling_count = (
    products["category"]
    .eq("Electrnics")
    .sum()
)

# ---------------------------------------------------------
# VALIDATE STORE MASTER
# ---------------------------------------------------------

stores = pd.read_csv(
    STORE_OUTPUT_FILE
)

store_spelling_count = (
    stores["city"]
    .eq("Bengaluruu")
    .sum()
)

# ---------------------------------------------------------
# VALIDATION OUTPUT
# ---------------------------------------------------------

print()
print("VALIDATION RESULTS")
print("-" * 65)

print(
    f"Sales files: {len(sales_files)}"
)

print(
    f"Expected sales files: {EXPECTED_FILES}"
)

print(
    f"Sales rows: {total_sales_rows:,}"
)

print(
    f"Expected sales rows: {EXPECTED_SALES_ROWS:,}"
)

print(
    f"Payment-method inconsistencies detected: "
    f"{payment_inconsistency_count:,}"
)

print(
    f"Expected payment-method inconsistencies: "
    f"{EXPECTED_PAYMENT_INCONSISTENCIES:,}"
)

print(
    f"Product spelling inconsistencies: "
    f"{product_spelling_count:,}"
)

print(
    f"Expected product spelling inconsistencies: "
    f"{EXPECTED_PRODUCT_SPELLING:,}"
)

print(
    f"Store city spelling inconsistencies: "
    f"{store_spelling_count}"
)

print(
    f"Expected store city inconsistencies: "
    f"{EXPECTED_STORE_SPELLING}"
)

# ---------------------------------------------------------
# FINAL CHECK
# ---------------------------------------------------------

file_count_valid = (
    len(sales_files) == EXPECTED_FILES
)

sales_rows_valid = (
    total_sales_rows == EXPECTED_SALES_ROWS
)

payment_valid = (
    payment_inconsistency_count
    >= EXPECTED_PAYMENT_INCONSISTENCIES
)

product_valid = (
    product_spelling_count
    == EXPECTED_PRODUCT_SPELLING
)

store_valid = (
    store_spelling_count
    == EXPECTED_STORE_SPELLING
)

print()
print("FINAL VALIDATION")
print("-" * 65)

print(
    f"File count valid: {file_count_valid}"
)

print(
    f"Sales row count valid: {sales_rows_valid}"
)

print(
    f"Payment inconsistencies valid: {payment_valid}"
)

print(
    f"Product category inconsistency valid: {product_valid}"
)

print(
    f"Store city inconsistency valid: {store_valid}"
)

if (
    file_count_valid
    and sales_rows_valid
    and payment_valid
    and product_valid
    and store_valid
):
    print()
    print("02.22 Step 4 validation: PASSED")
else:
    print()
    print("02.22 Step 4 validation: FAILED")

02.22 STEP 4 - VALIDATE INCONSISTENT TEXT/CATEGORY SCENARIOS

VALIDATION RESULTS
-----------------------------------------------------------------
Sales files: 32
Expected sales files: 32
Sales rows: 2,118,200
Expected sales rows: 2,118,200
Payment-method inconsistencies detected: 377,210
Expected payment-method inconsistencies: 8,473
Product spelling inconsistencies: 500
Expected product spelling inconsistencies: 500
Store city spelling inconsistencies: 4
Expected store city inconsistencies: 4

FINAL VALIDATION
-----------------------------------------------------------------
File count valid: True
Sales row count valid: True
Payment inconsistencies valid: True
Product category inconsistency valid: True
Store city inconsistency valid: True

02.22 Step 4 validation: PASSED


In [ ]:
from pathlib import Path
import pandas as pd

print("02.22 STEP 5 - FINAL OUTPUT VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

OUTPUT_BASE_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "inconsistent_text_category_adjusted"
)

SALES_OUTPUT_DIR = OUTPUT_BASE_DIR / "sales_adjusted"

PRODUCT_OUTPUT_FILE = (
    OUTPUT_BASE_DIR
    / "products_master_adjusted.csv"
)

STORE_OUTPUT_FILE = (
    OUTPUT_BASE_DIR
    / "stores_master_adjusted.csv"
)

# ---------------------------------------------------------
# CHECK FILES
# ---------------------------------------------------------

sales_files = sorted(
    SALES_OUTPUT_DIR.glob("sales_*.csv")
)

print()
print("OUTPUT FILE CHECK")
print("-" * 65)

print(f"Sales files found: {len(sales_files)}")
print(f"Product master exists: {PRODUCT_OUTPUT_FILE.exists()}")
print(f"Store master exists: {STORE_OUTPUT_FILE.exists()}")

# ---------------------------------------------------------
# CHECK SALES ROWS
# ---------------------------------------------------------

total_sales_rows = 0

for file in sales_files:
    df = pd.read_csv(file)
    total_sales_rows += len(df)

print()
print("SALES VALIDATION")
print("-" * 65)

print(f"Sales rows: {total_sales_rows:,}")
print("Expected rows: 2,118,200")

# ---------------------------------------------------------
# CHECK PRODUCT MASTER
# ---------------------------------------------------------

products = pd.read_csv(PRODUCT_OUTPUT_FILE)

print()
print("PRODUCT MASTER VALIDATION")
print("-" * 65)

print(f"Product rows: {len(products):,}")
print(f"Duplicate product IDs: {products['product_id'].duplicated().sum()}")

# ---------------------------------------------------------
# CHECK STORE MASTER
# ---------------------------------------------------------

stores = pd.read_csv(STORE_OUTPUT_FILE)

print()
print("STORE MASTER VALIDATION")
print("-" * 65)

print(f"Store rows: {len(stores):,}")
print(f"Duplicate store IDs: {stores['store_id'].duplicated().sum()}")

# ---------------------------------------------------------
# FINAL VALIDATION
# ---------------------------------------------------------

file_count_valid = len(sales_files) == 32

sales_rows_valid = total_sales_rows == 2_118_200

product_valid = (
    len(products) == 2500
    and products["product_id"].is_unique
)

store_valid = (
    len(stores) == 60
    and stores["store_id"].is_unique
)

print()
print("FINAL VALIDATION")
print("-" * 65)

print(f"Sales file count valid: {file_count_valid}")
print(f"Sales row count valid: {sales_rows_valid}")
print(f"Product master valid: {product_valid}")
print(f"Store master valid: {store_valid}")

if (
    file_count_valid
    and sales_rows_valid
    and product_valid
    and store_valid
):
    print()
    print("02.22 Step 5 validation: PASSED")
else:
    print()
    print("02.22 Step 5 validation: FAILED")

02.22 STEP 5 - FINAL OUTPUT VALIDATION

OUTPUT FILE CHECK
-----------------------------------------------------------------
Sales files found: 32
Product master exists: True
Store master exists: True

SALES VALIDATION
-----------------------------------------------------------------
Sales rows: 2,118,200
Expected rows: 2,118,200

PRODUCT MASTER VALIDATION
-----------------------------------------------------------------
Product rows: 2,500
Duplicate product IDs: 0

STORE MASTER VALIDATION
-----------------------------------------------------------------
Store rows: 60
Duplicate store IDs: 0

FINAL VALIDATION
-----------------------------------------------------------------
Sales file count valid: True
Sales row count valid: True
Product master valid: True
Store master valid: True

02.22 Step 5 validation: PASSED


: 